In [55]:
from langgraph.graph import START,END,StateGraph
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
from typing import TypedDict
import os
import re


In [56]:
# Model creation
gemini_model = ChatGoogleGenerativeAI(
    model = 'gemini-3.6-flash',
    api_key = os.environ['Gemini_api'],
    temperature = 0
)

qwen_model = ChatOllama(
    model = 'qwen2.5-1.5b',
    temperature=0
)

In [57]:
@tool
def ticket_booking(user: str) -> str:
    """Book movie tickets based on user query"""

    ticket = [0,1,2,3,4,5,6,7]
    capacity = 10

    # Extract ticket quantity
    match = re.search(r'(\d+)\s*tickets?', user, re.IGNORECASE)

    quantity = int(match.group(1)) if match else 1

    available = capacity - len(ticket)

    if quantity <= available:
        ticket.extend([1] * quantity)

        return f"{quantity} Tickets Booked Successfully!"

    else:
        return f"Only {available} Tickets Available!"

In [ ]:
class AgentState(TypedDict):
    user : str
    decision : str
    tool_result : str
    answer : str

def agent(state: AgentState):

    user = state["user"]

    prompt = f"""
    You are an AI Agent.

    Analyze the user query: {user}

    If the query is related to movie ticket booking,
    respond exactly: USE_TOOL

    Otherwise respond exactly: DIRECT_ANSWER
    """

    response = gemini_model.invoke(prompt)

    content = response.content

    if isinstance(content, list):
        decision = "".join(
            item["text"]
            for item in content
            if isinstance(item, dict) and "text" in item
        )
    else:
        decision = str(content)

    return {
        "decision": decision.strip()
    }
def router(state: AgentState):

    if state["decision"] == "USE_TOOL":
        return "use_tool"

    return "final_answer"

def use_tool(state: AgentState):

    result = ticket_booking.invoke({
        "user": state["user"]
    })

    return {
        "tool_result": str(result)
    }

def direct_answer(state: AgentState):

    response = gemini_model.invoke(
        state["user"]
    )

    return {
        "answer": response.content
    }

def final_answer(state: AgentState):

    if state.get("tool_result"):

        return {
            "answer": state["tool_result"]
        }

    return direct_answer(state)



In [59]:
graph = StateGraph(AgentState)

graph.add_node("agent", agent)
graph.add_node("use_tool", use_tool)
graph.add_node("final_answer", final_answer)

graph.add_edge(START, "agent")

graph.add_conditional_edges(
    "agent",
    router,
    {
        "use_tool": "use_tool",
        "final_answer": "final_answer"
    }
)

graph.add_edge("use_tool", "final_answer")
graph.add_edge("final_answer", END)

app = graph.compile()

In [60]:
result = app.invoke({
    "user": "Book  movie 1 ticket for Leo",
    "decision": "",
    "tool_result": "",
    "answer": ""
})

print(result["answer"])

d:\Programs\Python\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


1 Tickets Booked Successfully!
